# Women's Outfit Recommender (Kaggle / Colab / Local)

Give **one women's product** (e.g. a t-shirt). The notebook picks **one complete set**:

**t-shirt + bag + pants + shoes**

Two output modes:
1. **Text** — short description of the outfit  
2. **Images** — show the selected items from **your** image folders

**Large catalogs (1000+ items):** embeddings are precomputed once, saved to disk, and advanced mode searches only the top-K matches per category (see section 2 settings).

---
### Data layout
```
modified-woman-fit-categorey-images/
├── tshirts/8757261/pic1.jpg
├── bags/1425337/pic1.jpg
├── pants/13946746/pic1.jpg
└── shoes/10159110/pic1.jpg
```
Each product has its own **ID folder** with `pic1.jpg` inside.

**Kaggle:** add [mahsamb/modified-woman-fit-categorey-images](https://www.kaggle.com/datasets/mahsamb/modified-woman-fit-categorey-images) via **Add Input**. Kaggle mounts it at `/kaggle/input/datasets/mahsamb/modified-woman-fit-categorey-images/`.

**Colab:** upload `modified-woman-fit-categorey-images.zip` when prompted in section 3.

**Local / GitHub:** unzip `modified-woman-fit-categorey-images.zip` next to this notebook (auto-detected in section 3), or place an extracted catalog folder there.

## 1. Install dependencies

In [ ]:
import importlib.util
import subprocess
import sys

def has_module(name: str) -> bool:
    return importlib.util.find_spec(name) is not None

needed = {
    "transformers": "transformers",
    "scikit-learn": "sklearn",
    "opencv-python-headless": "cv2",
    "ipywidgets": "ipywidgets",
}
missing = [pip_name for pip_name, mod in needed.items() if not has_module(mod)]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("Dependencies OK")

## 2. Settings — change these

In [ ]:
# --- USER SETTINGS ---

# "text"  -> print outfit description only
# "images" -> show selected product images (recommended)
OUTPUT_MODE = "images"  # "text" or "images"

# Leave None to auto-detect. On Kaggle the path is usually:
# /kaggle/input/datasets/mahsamb/modified-woman-fit-categorey-images
CATALOG_DIR = "modified-woman-fit-categorey-images"
IMAGE_ROOT = None

# Reference t-shirt — pick ONE:
#   "8757261"  = product ID from your tshirts/8757261/pic1.jpg folder
#   "upload"   = upload a new t-shirt photo at runtime (Colab only)
# Or skip editing this and use the dropdown in section 4b instead.
REFERENCE_TSHIRT = "8757261"

# Recommendation engine (default — or pick in section 4c dropdown):
#   "simple"   = fast; each item picked separately (35% color + 65% CLIP)
#   "advanced" = full outfit optimization (CLIP style + visual coherence + color)
RECOMMENDATION_MODE = "advanced"  # "simple" or "advanced"

# Optional: random seed for reproducible picks when scores tie
RANDOM_SEED = 42

# --- SCALING (large catalogs: hundreds / thousands of items) ---

# Save CLIP embeddings to disk so the next run skips re-encoding
USE_DISK_CACHE = True
EMBEDDING_CACHE_DIR = None  # None -> auto (local: inside catalog dir; Kaggle/Colab: working dir)

# Embed the full catalog once at startup (recommended for 100+ items)
PRECOMPUTE_CATALOG = True
EMBED_BATCH_SIZE = 32

# Advanced mode: search only top-K items per category (K³ combos, vectorized)
# e.g. 50 -> at most 50×50×50 = 125,000 outfit combos
TOP_K_ADVANCED = 50

# Simple mode: uses cached embeddings (no slow per-item CLIP pair calls)
TOP_K_SIMPLE = None  # None = score all items (fast with cache); e.g. 100 to cap

## 3. Load your image catalog

**Kaggle:** add **mahsamb/modified-woman-fit-categorey-images** via **Add Input** — loads `/kaggle/input/modified-woman-fit-categorey-images/`.

**Colab:** upload `modified-woman-fit-categorey-images.zip`.

**Local:** uses `./modified-woman-fit-categorey-images/` if present.

In [ ]:
import os
import zipfile
from pathlib import Path
from typing import List, Optional

CATALOG_FOLDERS = ("tshirts", "bags", "pants", "shoes")
CATEGORY_ALIASES_CHECK = {
    "tshirts": ["tshirts", "t-shirt", "t-shirts", "tshirt", "tops", "top"],
    "bags": ["bags", "bag", "handbags", "handbag", "purses", "purse"],
    "pants": ["pants", "trousers", "jeans", "bottoms", "bottom"],
    "shoes": ["shoes", "shoe", "footwear", "sneakers", "heels"],
}
CATALOG_DIR = globals().get("CATALOG_DIR", "modified-woman-fit-categorey-images")


def detect_environment() -> str:
    if Path("/kaggle/input").exists():
        return "kaggle"
    try:
        import google.colab  # noqa: F401

        return "colab"
    except ImportError:
        return "local"


def _subdir_names(root: Path) -> set:
    return {d.name.lower() for d in root.iterdir() if d.is_dir()}


def _has_catalog_layout(root: Path) -> bool:
    if not root.is_dir():
        return False
    subdirs = _subdir_names(root)
    return all(any(a in subdirs for a in aliases) for aliases in CATEGORY_ALIASES_CHECK.values())


def _find_catalog_by_rglob(kaggle_input: Path) -> Optional[Path]:
    hits = []
    for path in kaggle_input.rglob("*"):
        if not path.is_dir():
            continue
        if path.name.lower() not in CATEGORY_ALIASES_CHECK["tshirts"]:
            continue
        root = path.parent
        if _has_catalog_layout(root):
            hits.append(root)
    if hits:
        return sorted(hits, key=lambda p: len(str(p)))[0]
    return None


def _catalog_roots_under(base: Path) -> List[Path]:
    """Find catalog root at base or in any immediate subfolder."""
    found = []
    if not base.is_dir():
        return found
    if _has_catalog_layout(base):
        found.append(base)
    for child in sorted(base.iterdir()):
        if child.is_dir() and _has_catalog_layout(child):
            found.append(child)
    return found


def _kaggle_mount_candidates() -> List[Path]:
    kaggle_input = Path("/kaggle/input")
    candidates = []

    # New Kaggle layout: /kaggle/input/datasets/<user>/<dataset-slug>
    datasets_user = kaggle_input / "datasets" / "mahsamb"
    if datasets_user.is_dir():
        for child in sorted(datasets_user.iterdir()):
            if child.is_dir():
                candidates.append(child)

    # Classic layout: /kaggle/input/<dataset-slug>
    candidates.extend([
        kaggle_input / CATALOG_DIR,
        kaggle_input / "datasets" / "mahsamb" / CATALOG_DIR,
    ])

    if kaggle_input.is_dir():
        for child in sorted(kaggle_input.iterdir()):
            if child.is_dir() and child not in candidates:
                candidates.append(child)

    # de-duplicate while preserving order
    seen = set()
    unique = []
    for p in candidates:
        key = str(p)
        if key not in seen:
            seen.add(key)
            unique.append(p)
    return unique


def _kaggle_debug_listing() -> str:
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.exists():
        return "  /kaggle/input does not exist"

    lines = ["  Contents of /kaggle/input:"]

    def walk(path: Path, prefix: str, depth: int = 0):
        if depth > 3:
            return
        try:
            children = sorted(path.iterdir())
        except OSError:
            lines.append(f"{prefix}(unreadable)")
            return
        for child in children[:12]:
            if child.is_dir():
                subs = [d.name for d in child.iterdir() if d.is_dir()][:6] if depth < 3 else []
                lines.append(f"{prefix}{child.name}/ -> {subs}")
                if depth < 2:
                    walk(child, prefix + "  ", depth + 1)

    walk(kaggle_input, "    ")
    return "\n".join(lines)


ENV = detect_environment()
print(f"Environment: {ENV}")


def resolve_image_root(explicit: Optional[str] = None) -> str:
    if explicit:
        for root in _catalog_roots_under(Path(explicit)):
            print(f"Using explicit catalog: {root}")
            return str(root)

    if ENV == "kaggle":
        kaggle_input = Path("/kaggle/input")

        for mount in _kaggle_mount_candidates():
            for root in _catalog_roots_under(mount):
                print(f"Using Kaggle dataset: {root}")
                return str(root)

        found = _find_catalog_by_rglob(kaggle_input)
        if found:
            print(f"Using Kaggle dataset (deep search): {found}")
            return str(found)

        raise FileNotFoundError(
            "Could not find tshirts/bags/pants/shoes on Kaggle.\n"
            "Add mahsamb/modified-woman-fit-categorey-images via Input → Add Input.\n"
            "Expected path like:\n"
            "  /kaggle/input/datasets/mahsamb/modified-woman-fit-categorey-images/tshirts/...\n"
            + _kaggle_debug_listing()
        )

    if ENV == "colab":
        from google.colab import files

        print(f"Upload {CATALOG_DIR}.zip")
        uploaded = files.upload()
        zip_name = next(iter(uploaded))
        with zipfile.ZipFile(zip_name, "r") as z:
            z.extractall("/content")
        for root in _catalog_roots_under(Path("/content")):
            print(f"OK — extracted catalog: {root}")
            return str(root)
        raise FileNotFoundError(
            f"Zip must contain {CATALOG_DIR}/tshirts, bags, pants, shoes"
        )

    zip_candidates = [
        Path.cwd() / "modified-woman-fit-categorey-images.zip",
        Path.cwd() / "Modified Woman Fit Categorey Images.zip",
    ]
    for zip_path in zip_candidates:
        if zip_path.is_file():
            print(f"Extracting catalog zip: {zip_path.name}")
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(Path.cwd())
            break

    for candidate in [
        Path.cwd() / CATALOG_DIR,
        Path.cwd().parent / CATALOG_DIR,
        Path.cwd() / "Modified Woman Fit Categorey Images",
        Path(f"/content/{CATALOG_DIR}"),
    ]:
        for root in _catalog_roots_under(candidate):
            print(f"Using local catalog: {root.resolve()}")
            return str(root.resolve())

    raise FileNotFoundError(
        f"Catalog not found. Unzip modified-woman-fit-categorey-images.zip next to this notebook, "
        f"or place {CATALOG_DIR}/ (with tshirt/bag/pants/shoes folders) and set IMAGE_ROOT."
    )


IMAGE_ROOT = resolve_image_root(IMAGE_ROOT)
print(f"IMAGE_ROOT = {IMAGE_ROOT}")

## 4. Core recommender engine

In [ ]:
import random
import re
import hashlib
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import cv2
import numpy as np
import torch
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from transformers import CLIPModel, CLIPProcessor


def resolve_device() -> str:
    """Pick cuda only if PyTorch can actually run kernels on this GPU (Kaggle-safe)."""
    if not torch.cuda.is_available():
        return "cpu"
    try:
        x = torch.randn(64, 64, device="cuda")
        torch.mm(x, x)
        torch.cuda.synchronize()
        return "cuda"
    except Exception as exc:
        print(f"CUDA probe failed ({type(exc).__name__}) — using CPU instead")
        return "cpu"


DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

CATEGORY_ALIASES = {
    "tshirts": ["tshirts", "t-shirt", "t-shirts", "tshirt", "tops", "top"],
    "bags": ["bags", "bag", "handbags", "handbag", "purses", "purse"],
    "pants": ["pants", "trousers", "jeans", "bottoms", "bottom"],
    "shoes": ["shoes", "shoe", "footwear", "sneakers", "heels"],
}
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}

STYLE_PROMPTS = [
    "casual everyday women's outfit",
    "stylish modern women's fashion look",
    "coordinated women's streetwear outfit",
    "elegant well-matched women's clothing",
    "comfortable women's daily wear ensemble",
]

CATEGORY_PROMPTS = {
    "pants": "women's pants that complete a fashionable outfit",
    "shoes": "women's shoes that match the outfit style",
    "bags": "women's handbag that complements the whole look",
}


@dataclass
class OutfitItem:
    category: str
    path: Path
    dominant_colors: List[Tuple[int, int, int]]
    clip_score: float = 0.0


@dataclass
class OutfitSet:
    tshirt: OutfitItem
    bag: OutfitItem
    pants: OutfitItem
    shoes: OutfitItem
    total_score: float = 0.0
    score_breakdown: Optional[Dict[str, float]] = None
    mode: str = "simple"

    def as_dict(self) -> Dict[str, OutfitItem]:
        return {
            "t-shirt": self.tshirt,
            "bag": self.bag,
            "pants": self.pants,
            "shoes": self.shoes,
        }


def find_category_dirs(root: Path) -> Dict[str, Path]:
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"IMAGE_ROOT not found: {root}")

    subdirs = {d.name.lower(): d for d in root.iterdir() if d.is_dir()}
    resolved = {}
    for canonical, aliases in CATEGORY_ALIASES.items():
        for alias in aliases:
            if alias in subdirs:
                resolved[canonical] = subdirs[alias]
                break
    missing = [c for c in CATEGORY_ALIASES if c not in resolved]
    if missing:
        raise ValueError(
            f"Missing folders for: {missing}. Found subfolders: {list(subdirs.keys())}"
        )
    return resolved


def list_images(folder: Path) -> List[Path]:
    files = [p for p in folder.rglob("*") if p.suffix.lower() in IMAGE_EXTS and p.is_file()]
    if not files:
        raise ValueError(f"No images in {folder}")
    return sorted(files)


def extract_dominant_colors(image_path: Path, k: int = 3) -> List[Tuple[int, int, int]]:
    img = cv2.imread(str(image_path))
    if img is None:
        return [(128, 128, 128)]
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (120, 120))
    pixels = img.reshape(-1, 3).astype(np.float32)
    k = min(k, len(pixels))
    km = KMeans(n_clusters=k, n_init=3, random_state=RANDOM_SEED)
    km.fit(pixels)
    centers = km.cluster_centers_.astype(int)
    counts = np.bincount(km.labels_)
    order = counts.argsort()[::-1]
    return [tuple(map(int, centers[i])) for i in order]


def rgb_to_name(rgb: Tuple[int, int, int]) -> str:
    r, g, b = rgb
    names = [
        ((0, 0, 0), "black"),
        ((255, 255, 255), "white"),
        ((128, 128, 128), "gray"),
        ((200, 200, 200), "light gray"),
        ((50, 50, 50), "charcoal"),
        ((255, 0, 0), "red"),
        ((180, 30, 30), "burgundy"),
        ((255, 105, 180), "pink"),
        ((255, 192, 203), "soft pink"),
        ((0, 0, 255), "blue"),
        ((30, 60, 180), "navy"),
        ((100, 149, 237), "light blue"),
        ((0, 128, 0), "green"),
        ((144, 238, 144), "sage green"),
        ((255, 255, 0), "yellow"),
        ((255, 165, 0), "orange"),
        ((128, 0, 128), "purple"),
        ((210, 180, 140), "beige"),
        ((245, 222, 179), "cream"),
        ((139, 69, 19), "brown"),
        ((160, 82, 45), "tan"),
    ]
    best, best_dist = "multicolor", float("inf")
    for ref, name in names:
        d = (r - ref[0]) ** 2 + (g - ref[1]) ** 2 + (b - ref[2]) ** 2
        if d < best_dist:
            best_dist, best = d, name
    return best


def color_harmony_score(ref_colors: List[Tuple[int, int, int]], cand_colors: List[Tuple[int, int, int]]) -> float:
    """Higher = better color match (neutral-friendly, not identical)."""
    ref = np.array(ref_colors[:2], dtype=float)
    cand = np.array(cand_colors[:2], dtype=float)
    dist = np.linalg.norm(ref.mean(0) - cand.mean(0))
    # sweet spot: related but not clone
    harmony = 1.0 / (1.0 + abs(dist - 55.0) / 40.0)
    neutrals = {"black", "white", "gray", "charcoal", "beige", "cream", "tan", "navy"}
    ref_names = {rgb_to_name(tuple(map(int, c))) for c in ref}
    cand_names = {rgb_to_name(tuple(map(int, c))) for c in cand}
    if ref_names & neutrals or cand_names & neutrals:
        harmony += 0.15
    return float(harmony)


class OutfitRecommender:
    CACHE_VERSION = "v1"
    CLIP_MODEL_ID = "openai/clip-vit-base-patch32"

    SIMPLE_PROMPTS = {
        "pants": "women's pants that match this casual t-shirt outfit",
        "shoes": "women's shoes that go with this t-shirt and pants outfit",
        "bags": "women's handbag that completes this everyday outfit",
    }

    def __init__(self, image_root: str):
        self.root = Path(image_root)
        self.device = DEVICE
        self.dirs = find_category_dirs(self.root)
        self.catalog = {k: list_images(v) for k, v in self.dirs.items()}
        print("Catalog sizes:", {k: len(v) for k, v in self.catalog.items()})

        print("Loading CLIP...")
        self.model = CLIPModel.from_pretrained(self.CLIP_MODEL_ID).to(self.device)
        self.processor = CLIPProcessor.from_pretrained(self.CLIP_MODEL_ID)
        self.model.eval()
        self._ensure_clip_runs()
        self._image_embed_cache: Dict[Path, np.ndarray] = {}
        self._color_cache: Dict[Path, List[Tuple[int, int, int]]] = {}
        self._style_text_embeds: Optional[np.ndarray] = None
        self._category_text_embeds: Dict[str, np.ndarray] = {}
        self._simple_prompt_embeds: Dict[str, np.ndarray] = {}

        self._load_disk_cache()
        if PRECOMPUTE_CATALOG:
            self.precompute_catalog(save=True)

    def _get_cache_dir(self) -> Path:
        if EMBEDDING_CACHE_DIR:
            return Path(EMBEDDING_CACHE_DIR)
        # Kaggle/Colab input folders are often read-only — cache in working dir
        if ENV == "kaggle":
            return Path("/kaggle/working") / ".embedding_cache" / self.root.name
        if ENV == "colab":
            return Path("/content") / ".embedding_cache" / self.root.name
        return self.root / ".embedding_cache"

    def _catalog_fingerprint(self) -> str:
        parts = []
        for cat in sorted(self.catalog):
            for p in sorted(self.catalog[cat], key=str):
                rel = p.relative_to(self.root)
                stat = p.stat()
                parts.append(f"{rel}:{stat.st_mtime_ns}:{stat.st_size}")
        payload = "\n".join(parts)
        return hashlib.sha256(payload.encode()).hexdigest()[:16]

    def _cache_file(self) -> Path:
        return self._get_cache_dir() / f"{self._catalog_fingerprint()}_{self.CACHE_VERSION}.npz"

    def _missing_catalog_paths(self) -> List[Path]:
        all_paths = []
        for cat in self.catalog:
            all_paths.extend(self.catalog[cat])
        return [p for p in all_paths if p not in self._image_embed_cache]

    def _load_disk_cache(self) -> bool:
        if not USE_DISK_CACHE:
            return False
        cache_file = self._cache_file()
        if not cache_file.exists():
            print("No disk embedding cache found — will compute on first run.")
            return False

        data = np.load(cache_file, allow_pickle=True)
        if str(data.get("model", self.CLIP_MODEL_ID)) != self.CLIP_MODEL_ID:
            print("Disk cache model mismatch — recomputing embeddings.")
            return False

        loaded = 0
        for rel_path, emb in zip(data["paths"], data["embeddings"]):
            path = self.root / str(rel_path)
            if path.exists():
                self._image_embed_cache[path] = np.asarray(emb, dtype=np.float32)
                loaded += 1

        if "colors" in data:
            for rel_path, colors in zip(data["paths"], data["colors"]):
                path = self.root / str(rel_path)
                if path.exists():
                    self._color_cache[path] = [tuple(map(int, c)) for c in colors]

        print(f"Loaded {loaded} embeddings from {cache_file}")
        return loaded > 0

    def _save_disk_cache(self) -> None:
        if not USE_DISK_CACHE or not self._image_embed_cache:
            return

        cache_dir = self._get_cache_dir()
        cache_dir.mkdir(parents=True, exist_ok=True)
        cache_file = self._cache_file()

        paths = sorted(self._image_embed_cache.keys(), key=str)
        rel_paths = np.array([str(p.relative_to(self.root)) for p in paths])
        embeddings = np.stack([self._image_embed_cache[p] for p in paths]).astype(np.float32)
        colors = np.array(
            [self._get_colors(p) for p in paths],
            dtype=object,
        )

        np.savez_compressed(
            cache_file,
            paths=rel_paths,
            embeddings=embeddings,
            colors=colors,
            model=self.CLIP_MODEL_ID,
            fingerprint=self._catalog_fingerprint(),
        )
        print(f"Saved {len(paths)} embeddings to {cache_file}")

    def _get_colors(self, path: Path) -> List[Tuple[int, int, int]]:
        if path not in self._color_cache:
            self._color_cache[path] = extract_dominant_colors(path)
        return self._color_cache[path]

    def precompute_catalog(self, save: bool = True) -> None:
        all_paths = []
        for cat in self.catalog:
            all_paths.extend(self.catalog[cat])

        missing = [p for p in all_paths if p not in self._image_embed_cache]
        if not missing:
            print("Catalog embeddings already in memory.")
            return

        print(f"Precomputing embeddings for {len(missing)} images (batch={EMBED_BATCH_SIZE})...")
        self._embed_images_batch(missing, batch_size=EMBED_BATCH_SIZE)

        for p in missing:
            self._get_colors(p)

        if save:
            self._save_disk_cache()
        print("Catalog precompute done.")

    @staticmethod
    def _as_feature_tensor(feats) -> torch.Tensor:
        """Normalize CLIP outputs across transformers versions."""
        if isinstance(feats, torch.Tensor):
            return feats
        for attr in ("image_embeds", "text_embeds", "pooler_output"):
            if hasattr(feats, attr):
                val = getattr(feats, attr)
                if val is not None:
                    return val
        if hasattr(feats, "last_hidden_state"):
            return feats.last_hidden_state[:, 0]
        raise TypeError(f"Unexpected CLIP feature type: {type(feats)}")

    @staticmethod
    def _normalize(feats: torch.Tensor) -> torch.Tensor:
        return feats / feats.norm(dim=-1, keepdim=True)

    @staticmethod
    def _pooler_output(outputs, input_ids: Optional[torch.Tensor] = None) -> torch.Tensor:
        pooled = getattr(outputs, "pooler_output", None)
        if pooled is not None:
            return pooled
        hidden = outputs.last_hidden_state
        if input_ids is not None:
            eos = input_ids.argmax(dim=-1)
            return hidden[torch.arange(hidden.shape[0], device=hidden.device), eos]
        return hidden[:, 0]

    @torch.no_grad()
    def _encode_images(self, pixel_values: torch.Tensor) -> torch.Tensor:
        vision_outputs = self.model.vision_model(pixel_values=pixel_values)
        pooled = self._pooler_output(vision_outputs)
        return self.model.visual_projection(pooled)

    @torch.no_grad()
    def _encode_texts(self, input_ids: torch.Tensor, attention_mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        text_outputs = self.model.text_model(input_ids=input_ids, attention_mask=attention_mask)
        pooled = self._pooler_output(text_outputs, input_ids=input_ids)
        return self.model.text_projection(pooled)

    def _ensure_clip_runs(self) -> None:
        """Run tiny CLIP text + vision passes; fall back to CPU if GPU kernels fail."""
        global DEVICE
        dummy = Image.new("RGB", (224, 224), color=(128, 128, 128))

        def _probe() -> None:
            with torch.no_grad():
                text_in = self.processor(text=["fashion"], return_tensors="pt", padding=True).to(self.device)
                self._normalize(self._encode_texts(text_in["input_ids"], text_in.get("attention_mask")))
                img_in = self.processor(images=[dummy], return_tensors="pt", padding=True).to(self.device)
                self._normalize(self._encode_images(img_in["pixel_values"]))

        try:
            _probe()
        except Exception as exc:
            if self.device == "cpu":
                raise
            print(f"CLIP failed on GPU ({type(exc).__name__}) — switching to CPU")
            self.device = "cpu"
            DEVICE = "cpu"
            self.model = self.model.to("cpu")
            _probe()
            print("Using device: cpu")

    @torch.no_grad()
    def _embed_images_batch(
        self, paths: List[Path], batch_size: Optional[int] = None
    ) -> Dict[Path, np.ndarray]:
        batch_size = batch_size or EMBED_BATCH_SIZE
        missing = [p for p in paths if p not in self._image_embed_cache]
        for i in range(0, len(missing), batch_size):
            batch_paths = missing[i : i + batch_size]
            images = [Image.open(p).convert("RGB") for p in batch_paths]
            inputs = self.processor(images=images, return_tensors="pt", padding=True).to(self.device)
            feats = self._normalize(self._encode_images(inputs["pixel_values"]))
            for path, vec in zip(batch_paths, feats.cpu().numpy()):
                self._image_embed_cache[path] = vec
        return {p: self._image_embed_cache[p] for p in paths}

    @torch.no_grad()
    def _embed_texts(self, texts: List[str]) -> np.ndarray:
        inputs = self.processor(text=texts, return_tensors="pt", padding=True).to(self.device)
        feats = self._normalize(
            self._encode_texts(inputs["input_ids"], inputs.get("attention_mask"))
        )
        return feats.cpu().numpy()

    def _ensure_text_embeddings(self):
        if self._style_text_embeds is None:
            self._style_text_embeds = self._embed_texts(STYLE_PROMPTS)
        for cat, prompt in CATEGORY_PROMPTS.items():
            if cat not in self._category_text_embeds:
                self._category_text_embeds[cat] = self._embed_texts([prompt])[0]

    def _ensure_simple_prompt_embeds(self):
        for cat, prompt in self.SIMPLE_PROMPTS.items():
            if cat not in self._simple_prompt_embeds:
                self._simple_prompt_embeds[cat] = self._embed_texts([prompt])[0]

    def _fast_pair_score(
        self,
        t_emb: np.ndarray,
        c_emb: np.ndarray,
        prompt_emb: np.ndarray,
    ) -> float:
        """CLIP-like score using precomputed embeddings (no extra model calls)."""
        sim_a = self._cosine(t_emb, prompt_emb)
        sim_b = self._cosine(c_emb, prompt_emb)
        cross = self._cosine(t_emb, c_emb)
        return 0.45 * sim_a + 0.45 * sim_b + 0.10 * cross

    def _top_k_paths(
        self,
        t_emb: np.ndarray,
        paths: List[Path],
        embeds: Dict[Path, np.ndarray],
        k: int,
    ) -> List[Path]:
        if k is None or k >= len(paths):
            return list(paths)
        scored = [(p, self._cosine(t_emb, embeds[p])) for p in paths]
        scored.sort(key=lambda x: x[1], reverse=True)
        return [p for p, _ in scored[:k]]

    @staticmethod
    def _cosine(a: np.ndarray, b: np.ndarray) -> float:
        return float(np.dot(a, b))

    def _style_score(self, image_emb: np.ndarray) -> float:
        self._ensure_text_embeddings()
        return float(np.mean(self._style_text_embeds @ image_emb))

    def _category_score(self, image_emb: np.ndarray, category: str) -> float:
        self._ensure_text_embeddings()
        return self._cosine(image_emb, self._category_text_embeds[category])

    @torch.no_grad()
    def clip_pair_score(self, img_a: Path, img_b: Path, prompt: str) -> float:
        a = Image.open(img_a).convert("RGB")
        b = Image.open(img_b).convert("RGB")
        inputs = self.processor(
            text=[prompt],
            images=[a, b],
            return_tensors="pt",
            padding=True,
        ).to(self.device)
        txt_feat = self._normalize(
            self._encode_texts(inputs["input_ids"], inputs.get("attention_mask"))
        )
        img_feats = self._normalize(self._encode_images(inputs["pixel_values"]))
        sim_a = (img_feats[0:1] @ txt_feat.T).item()
        sim_b = (img_feats[1:2] @ txt_feat.T).item()
        cross = (img_feats[0:1] @ img_feats[1:2].T).item()
        return 0.45 * sim_a + 0.45 * sim_b + 0.10 * cross

    def resolve_tshirt(self, reference: str) -> Path:
        if reference == "upload":
            if ENV == "colab":
                from google.colab import files

                print("Upload your reference t-shirt image")
                up = files.upload()
                ref_path = Path("/content") / next(iter(up))
                return ref_path
            raise ValueError(
                "REFERENCE_TSHIRT='upload' works on Colab only. "
                "On Kaggle, set a product ID like '8757261'."
            )
        ref = Path(reference)
        if ref.is_file():
            return ref

        pid = reference.strip()
        # Match your layout: tshirts/8757261/pic1.jpg
        id_matches = [p for p in self.catalog["tshirts"] if p.parent.name == pid]
        if id_matches:
            return id_matches[0]

        name_matches = [p for p in self.catalog["tshirts"] if p.name == reference]
        if name_matches:
            return name_matches[0]

        raise FileNotFoundError(
            f"T-shirt not found: {reference}. "
            f"Use a product ID like '8757261' or set REFERENCE_TSHIRT='upload'"
        )

    def _rank_candidates(
        self,
        tshirt_path: Path,
        category: str,
        prompt: str,
        ref_colors: List[Tuple[int, int, int]],
        top_k: int = 5,
        t_emb: Optional[np.ndarray] = None,
        embeds: Optional[Dict[Path, np.ndarray]] = None,
    ) -> List[OutfitItem]:
        self._ensure_simple_prompt_embeds()
        prompt_key = category if category in self.SIMPLE_PROMPTS else None
        prompt_emb = self._simple_prompt_embeds.get(prompt_key) if prompt_key else self._embed_texts([prompt])[0]

        if t_emb is None or embeds is None:
            embeds = self._embed_images_batch([tshirt_path] + self.catalog[category])
            t_emb = embeds[tshirt_path]

        candidate_paths = self._top_k_paths(
            t_emb,
            self.catalog[category],
            embeds,
            TOP_K_SIMPLE,
        )

        scored = []
        for p in candidate_paths:
            colors = self._get_colors(p)
            c_score = color_harmony_score(ref_colors, colors)
            clip_score = self._fast_pair_score(t_emb, embeds[p], prompt_emb)
            total = 0.35 * c_score + 0.65 * clip_score
            scored.append(OutfitItem(category, p, colors, total))
        scored.sort(key=lambda x: x.clip_score, reverse=True)
        return scored[:top_k]

    def recommend_simple(self, tshirt_path: Path) -> OutfitSet:
        random.seed(RANDOM_SEED)
        ref_colors = self._get_colors(tshirt_path)

        all_paths = (
            [tshirt_path]
            + self.catalog["pants"]
            + self.catalog["shoes"]
            + self.catalog["bags"]
        )
        embeds = self._embed_images_batch(all_paths, batch_size=EMBED_BATCH_SIZE)
        t_emb = embeds[tshirt_path]

        tshirt_item = OutfitItem("tshirts", tshirt_path, ref_colors, 1.0)

        pants_rank = self._rank_candidates(
            tshirt_path,
            "pants",
            self.SIMPLE_PROMPTS["pants"],
            ref_colors,
            t_emb=t_emb,
            embeds=embeds,
        )
        pants = pants_rank[0]

        shoes_rank = self._rank_candidates(
            tshirt_path,
            "shoes",
            self.SIMPLE_PROMPTS["shoes"],
            ref_colors,
            t_emb=t_emb,
            embeds=embeds,
        )
        shoes = shoes_rank[0]

        bag_rank = self._rank_candidates(
            tshirt_path,
            "bags",
            self.SIMPLE_PROMPTS["bags"],
            ref_colors,
            t_emb=t_emb,
            embeds=embeds,
        )
        bag = bag_rank[0]

        return OutfitSet(
            tshirt=tshirt_item,
            bag=bag,
            pants=pants,
            shoes=shoes,
            total_score=(pants.clip_score + shoes.clip_score + bag.clip_score) / 3,
            score_breakdown={
                "pants_match": pants.clip_score,
                "shoes_match": shoes.clip_score,
                "bag_match": bag.clip_score,
                "color_weight": 0.35,
                "clip_weight": 0.65,
                "used_cached_embeddings": 1.0,
            },
            mode="simple",
        )

    def _score_outfit_combos_vectorized(
        self,
        t_emb: np.ndarray,
        ref_colors: List[Tuple[int, int, int]],
        pants_paths: List[Path],
        shoes_paths: List[Path],
        bags_paths: List[Path],
        embeds: Dict[Path, np.ndarray],
    ) -> Tuple[float, Tuple, Dict[str, float]]:
        p_embs = np.stack([embeds[p] for p in pants_paths])
        s_embs = np.stack([embeds[p] for p in shoes_paths])
        b_embs = np.stack([embeds[p] for p in bags_paths])

        style_t = self._style_score(t_emb)
        style_p = np.array([self._style_score(e) for e in p_embs])
        style_s = np.array([self._style_score(e) for e in s_embs])
        style_b = np.array([self._style_score(e) for e in b_embs])

        cat_p = np.array([self._category_score(e, "pants") for e in p_embs])
        cat_s = np.array([self._category_score(e, "shoes") for e in s_embs])
        cat_b = np.array([self._category_score(e, "bags") for e in b_embs])

        col_p = np.array([color_harmony_score(ref_colors, self._get_colors(p)) for p in pants_paths])
        col_s = np.array([color_harmony_score(ref_colors, self._get_colors(p)) for p in shoes_paths])
        col_b = np.array([color_harmony_score(ref_colors, self._get_colors(p)) for p in bags_paths])

        anchor_p = p_embs @ t_emb
        anchor_s = s_embs @ t_emb
        anchor_b = b_embs @ t_emb
        anchor = (anchor_p[:, None, None] + anchor_s[None, :, None] + anchor_b[None, None, :]) / 3.0

        ps = p_embs @ s_embs.T
        pb = p_embs @ b_embs.T
        sb = s_embs @ b_embs.T
        coherence = (ps[:, :, None] + pb[:, None, :] + sb[None, :, :]) / 3.0

        style = (
            style_t
            + style_p[:, None, None]
            + style_s[None, :, None]
            + style_b[None, None, :]
        ) / 4.0

        category_fit = (
            cat_p[:, None, None] + cat_s[None, :, None] + cat_b[None, None, :]
        ) / 3.0

        color_h = (
            col_p[:, None, None] + col_s[None, :, None] + col_b[None, None, :]
        ) / 3.0

        total = (
            0.30 * anchor
            + 0.28 * coherence
            + 0.22 * style
            + 0.12 * category_fit
            + 0.08 * color_h
        )

        flat_idx = int(np.argmax(total))
        pi, si, bi = np.unravel_index(flat_idx, total.shape)
        best_score = float(total[pi, si, bi])

        pants_path = pants_paths[pi]
        shoes_path = shoes_paths[si]
        bag_path = bags_paths[bi]

        best_breakdown = {
            "anchor_visual": float(anchor[pi, si, bi]),
            "outfit_coherence": float(coherence[pi, si, bi]),
            "style_alignment": float(style[pi, si, bi]),
            "category_fit": float(category_fit[pi, si, bi]),
            "color_harmony": float(color_h[pi, si, bi]),
            "total": best_score,
            "search_space": float(total.size),
        }

        best = (
            pants_path,
            shoes_path,
            bag_path,
            self._get_colors(pants_path),
            self._get_colors(shoes_path),
            self._get_colors(bag_path),
        )
        return best_score, best, best_breakdown

    def recommend_advanced(self, tshirt_path: Path) -> OutfitSet:
        """Joint outfit search with top-K prefilter + vectorized scoring."""
        random.seed(RANDOM_SEED)
        ref_colors = self._get_colors(tshirt_path)

        all_paths = (
            [tshirt_path]
            + self.catalog["pants"]
            + self.catalog["shoes"]
            + self.catalog["bags"]
        )
        embeds = self._embed_images_batch(all_paths, batch_size=EMBED_BATCH_SIZE)
        t_emb = embeds[tshirt_path]

        pants_paths = self._top_k_paths(t_emb, self.catalog["pants"], embeds, TOP_K_ADVANCED)
        shoes_paths = self._top_k_paths(t_emb, self.catalog["shoes"], embeds, TOP_K_ADVANCED)
        bags_paths = self._top_k_paths(t_emb, self.catalog["bags"], embeds, TOP_K_ADVANCED)

        combos = len(pants_paths) * len(shoes_paths) * len(bags_paths)
        print(
            f"Advanced search: top-{len(pants_paths)} pants × "
            f"{len(shoes_paths)} shoes × {len(bags_paths)} bags = {combos:,} combos"
        )

        best_score, best, best_breakdown = self._score_outfit_combos_vectorized(
            t_emb,
            ref_colors,
            pants_paths,
            shoes_paths,
            bags_paths,
            embeds,
        )

        pants_path, shoes_path, bag_path, p_colors, s_colors, b_colors = best

        return OutfitSet(
            tshirt=OutfitItem("tshirts", tshirt_path, ref_colors, 1.0),
            pants=OutfitItem("pants", pants_path, p_colors, best_breakdown["anchor_visual"]),
            shoes=OutfitItem("shoes", shoes_path, s_colors, best_breakdown["outfit_coherence"]),
            bag=OutfitItem("bags", bag_path, b_colors, best_breakdown["category_fit"]),
            total_score=best_score,
            score_breakdown=best_breakdown,
            mode="advanced",
        )

    def recommend(self, tshirt_path: Path, mode: Optional[str] = None) -> OutfitSet:
        mode = (mode or RECOMMENDATION_MODE).lower()
        if mode == "advanced":
            print(
                f"Using ADVANCED mode: top-{TOP_K_ADVANCED} prefilter + "
                "vectorized CLIP outfit search"
            )
            return self.recommend_advanced(tshirt_path)
        if mode == "simple":
            print("Using SIMPLE mode: cached embeddings (35% color + 65% CLIP-like score)")
            return self.recommend_simple(tshirt_path)
        raise ValueError('RECOMMENDATION_MODE must be "simple" or "advanced"')


def product_id(path: Path) -> str:
    """Your folders: category/PRODUCT_ID/pic1.jpg"""
    if path.parent.name.isdigit():
        return path.parent.name
    return path.stem


def human_name(path: Path) -> str:
    pid = product_id(path)
    if pid.isdigit():
        return f"Product {pid}"
    stem = path.stem.replace("_", " ").replace("-", " ")
    stem = re.sub(r"\s+", " ", stem).strip()
    return stem.title() if stem else path.name


def describe_item(item: OutfitItem) -> str:
    main_color = rgb_to_name(item.dominant_colors[0])
    accent = rgb_to_name(item.dominant_colors[1]) if len(item.dominant_colors) > 1 else main_color
    return f"{main_color} ({accent} accent) — {human_name(item.path)}"


def outfit_to_text(outfit: OutfitSet) -> str:
    lines = [
        "=== Recommended Women's Outfit Set ===",
        f"Mode: {outfit.mode.upper()} | Total score: {outfit.total_score:.4f}",
        "",
        f"1. T-shirt: {describe_item(outfit.tshirt)}",
        f"2. Pants:   {describe_item(outfit.pants)}",
        f"3. Shoes:   {describe_item(outfit.shoes)}",
        f"4. Bag:     {describe_item(outfit.bag)}",
        "",
        f"T-shirt ID: {product_id(outfit.tshirt.path)} | "
        f"Pants: {product_id(outfit.pants.path)} | "
        f"Shoes: {product_id(outfit.shoes.path)} | "
        f"Bag: {product_id(outfit.bag.path)}",
    ]

    if outfit.score_breakdown:
        lines.append("")
        lines.append("Score breakdown:")
        for key, value in outfit.score_breakdown.items():
            lines.append(f"  - {key}: {value:.4f}")

    if outfit.mode == "simple":
        lines.append("")
        lines.append(
            "Simple mode: cached embeddings + color (35% color + 65% CLIP-like score)."
        )
    else:
        lines.append("")
        lines.append(
            f"Advanced mode: top-{TOP_K_ADVANCED} prefilter, vectorized joint search "
            "(CLIP visual/style/coherence; color ~8%)."
        )
        if outfit.score_breakdown and "search_space" in outfit.score_breakdown:
            lines.append(
                f"  Search space: {int(outfit.score_breakdown['search_space']):,} outfit combos"
            )

    return "\n".join(lines)


def show_outfit_images(outfit: OutfitSet):
    items = outfit.as_dict()
    fig, axes = plt.subplots(1, 4, figsize=(16, 5))
    for ax, (label, item) in zip(axes, items.items()):
        img = Image.open(item.path).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"{label.title()}\n{human_name(item.path)}", fontsize=10)
        ax.axis("off")
    plt.suptitle("Your recommended outfit set", fontsize=14, fontweight="bold")
    plt.tight_layout()
    plt.show()


def check_my_data(root: str):
    root = Path(root)
    dirs = find_category_dirs(root)
    print("=== Your catalog check ===")
    for cat, folder in sorted(dirs.items()):
        imgs = list_images(folder)
        sample = imgs[0].relative_to(root)
        print(f"  {cat:8s} : {len(imgs):3d} products  (example: {sample})")
    print()


check_my_data(IMAGE_ROOT)

recommender = OutfitRecommender(IMAGE_ROOT)
print("Available t-shirt IDs (first 10):", [product_id(p) for p in recommender.catalog["tshirts"][:10]])
tshirt_path = recommender.resolve_tshirt(REFERENCE_TSHIRT)
recommendation_mode = RECOMMENDATION_MODE.lower()
print(f"Reference t-shirt: {tshirt_path}  (ID: {product_id(tshirt_path)})")
print(f"Recommendation mode: {recommendation_mode}  (change in section 4c)")

## 4b. (Optional) Pick t-shirt from dropdown

Run this cell to choose any t-shirt from your catalog with a dropdown + preview.  
Click **Use this t-shirt**, then run the recommendation cell below.

Skip this cell if you already set `REFERENCE_TSHIRT` in settings.

In [ ]:
from IPython.display import display, clear_output
import ipywidgets as widgets

tshirt_ids = sorted(product_id(p) for p in recommender.catalog["tshirts"])
default_id = REFERENCE_TSHIRT if REFERENCE_TSHIRT in tshirt_ids else tshirt_ids[0]

dropdown = widgets.Dropdown(
    options=tshirt_ids,
    value=default_id,
    description="T-shirt ID:",
    layout=widgets.Layout(width="420px"),
)

preview = widgets.Output()
status = widgets.Output()


def show_preview(product_id_value):
    with preview:
        clear_output(wait=True)
        path = recommender.resolve_tshirt(product_id_value)
        img = Image.open(path).convert("RGB")
        fig, ax = plt.subplots(figsize=(3.5, 3.5))
        ax.imshow(img)
        ax.set_title(f"Product {product_id_value}", fontsize=11)
        ax.axis("off")
        plt.show()


def on_dropdown_change(change):
    if change["name"] == "value":
        show_preview(change["new"])


def apply_selection(_=None):
    global tshirt_path
    tshirt_path = recommender.resolve_tshirt(dropdown.value)
    with status:
        clear_output(wait=True)
        print(f"Selected t-shirt ID: {dropdown.value}")
        print(f"Path: {tshirt_path}")


dropdown.observe(on_dropdown_change, names="value")
apply_btn = widgets.Button(description="Use this t-shirt", button_style="primary")
apply_btn.on_click(apply_selection)

display(widgets.VBox([
    widgets.HTML("<b>All t-shirts in your catalog:</b>"),
    dropdown,
    apply_btn,
    preview,
    status,
]))

show_preview(default_id)
apply_selection()

## 4c. Pick recommendation mode (Simple / Advanced)

Choose how outfits are built, then click **Use this mode** before running section 5.

| Mode | Best for |
|------|----------|
| **Simple** | Fast results; each item (pants, shoes, bag) matched to the t-shirt separately |
| **Advanced** | Better full-outfit coherence; searches top-K combos jointly (recommended) |

Skip this cell if you already set `RECOMMENDATION_MODE` in section 2.

In [ ]:
from IPython.display import display, clear_output
import ipywidgets as widgets

_mode_default = recommendation_mode if recommendation_mode in ("simple", "advanced") else "advanced"

mode_dropdown = widgets.Dropdown(
    options=[
        ("Simple — fast, pick each item separately", "simple"),
        ("Advanced — full outfit optimization (recommended)", "advanced"),
    ],
    value=_mode_default,
    description="Mode:",
    layout=widgets.Layout(width="520px"),
)

mode_status = widgets.Output()


def apply_mode(_=None):
    global recommendation_mode
    recommendation_mode = mode_dropdown.value
    with mode_status:
        clear_output(wait=True)
        if recommendation_mode == "simple":
            print("Selected: SIMPLE — cached embeddings, per-item matching")
        else:
            print(
                f"Selected: ADVANCED — top-{TOP_K_ADVANCED} prefilter + "
                "vectorized joint outfit search"
            )


mode_btn = widgets.Button(description="Use this mode", button_style="primary")
mode_btn.on_click(apply_mode)

display(widgets.VBox([
    widgets.HTML("<b>Recommendation engine:</b>"),
    mode_dropdown,
    mode_btn,
    mode_status,
]))

apply_mode()

## 5. Run recommendation

Uses the t-shirt from section 4 (or 4b) and the mode from section 4c (or section 2 settings).

In [ ]:
outfit = recommender.recommend(tshirt_path, mode=recommendation_mode)

if OUTPUT_MODE.lower() == "text":
    print(outfit_to_text(outfit))
elif OUTPUT_MODE.lower() == "images":
    show_outfit_images(outfit)
    print("\n" + outfit_to_text(outfit))
else:
    raise ValueError('OUTPUT_MODE must be "text" or "images"')

## 6. (Optional) Try another t-shirt from your catalog

In [ ]:
# Try another t-shirt by product ID from your catalog
OTHER_TSHIRT_ID = "8797011"  # change to any ID printed above

other_path = recommender.resolve_tshirt(OTHER_TSHIRT_ID)
outfit2 = recommender.recommend(other_path, mode=recommendation_mode)
show_outfit_images(outfit2)
print(outfit_to_text(outfit2))